# Assignment 2B — RAG Pipeline on the HR-policy corpus (Group 4)

A small, complete Retrieval-Augmented Generation system built on the Assignment 1 (CorpPolicyLM) domain corpus.

| Part | What is done | Marks |
|---|---|---|
| A | Dense search (FAISS) and choice of chunk size | 5 |
| B | Dense vs keyword (BM25) vs hybrid (RRF) search | 4 |
| C | Cross-encoder reranking | 4 |
| D | Grounded answer generation and an LLM judge | 7 |

**Reused from Assignment 1:**
* **Documents:** the 33 cleaned `.txt` policy documents (`data/domain_corpus/`, from 34 PDFs, 902 pages).
* **Queries:** 10 queries with expected answers from the Assignment 1 evaluation set (`data/a1_eval/instruction_eval.jsonl`).
* **Generator:** the fine-tuned Assignment 1 model (TinyLlama-1.1B CPT checkpoint + QLoRA Adapter B) is tried first in D1.

**Generator and judge used after the switch:** `Qwen2.5-3B-Instruct` generates and `Llama-3.2-3B-Instruct` judges, both pre-quantised to 4-bit. These are the 3B members of the two model families the brief names. The 7–8B versions (about 5.5 GB each) do not fit the lab server's 4.9 GB home disk, and the lab quota did not allow a server with a bigger volume. The code runs the 7–8B models unchanged when `GENERATOR` / `JUDGE` are set.

**Code:** the pipeline lives in `src/rag/`; this notebook calls it and shows every result:
* `data.py` — chunking and the hit test;
* `retrieval.py` — dense, BM25 and RRF search;
* `rerank.py` — cross-encoder reranking;
* `generation.py` — prompts, generator and judge.

In [1]:
import os, sys, json, time, inspect, importlib.util
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
missing = [m for m in ("faiss", "rank_bm25", "sentence_transformers", "peft", "bitsandbytes") if importlib.util.find_spec(m) is None]
if missing:
    raise SystemExit(f"Missing packages {missing}: in a terminal run `pip install -r requirements.txt`, then restart the kernel.")
import numpy as np, pandas as pd, torch, faiss, transformers, sentence_transformers
from IPython.display import display, Markdown
pd.set_option("display.max_colwidth", None)
from src.rag import config
from src.rag.data import load_corpus, load_queries, chunk_fixed, is_hit, answer_chunk_ids, norm
from src.rag import retrieval, rerank as rr, generation as gen
print("torch", torch.__version__, "| transformers", transformers.__version__, "| sentence-transformers",
      sentence_transformers.__version__, "| faiss", faiss.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")
print(f"Hugging Face cache: {gen.hf_cache_dir()} ({gen.free_gb():.1f} GiB free)")

torch 2.5.1+cu124 | transformers 5.18.0 | sentence-transformers 6.1.0 | faiss 1.9.0
GPU: NVIDIA A100-SXM4-80GB
Hugging Face cache: /home/jovyan/.cache/huggingface/hub (0.6 GiB free)


## Before you start — queries, answer spans and the embedding model's limit
The 10 queries come from the Assignment 1 evaluation set (12 held-out pairs from 6 documents). Two were left out:
* #1 asks about the same Niramai passage as #0 with a near-identical question.
* #3 is an abstract re-reading of the same Astral sentence as #2.

For each query, the **answer span** is the sentence or short passage in the corpus that answers it. Every span is checked to occur verbatim in its source document (`tools/make_queries.py`). A retrieved chunk is a **hit** if it contains the whole span; the comparison is case-insensitive and ignores line breaks.

A separate 11th query, whose answer is not in the corpus, is used only in D1.

In [2]:
docs = load_corpus()
queries, ooc = load_queries()
print(f"{len(docs)} documents, {sum(len(t.split()) for t in docs.values()):,} words")
display(pd.DataFrame(queries)[["qid", "query", "answer_span", "source", "expected_answer"]])
print("Out-of-corpus query (D1.3):", ooc["query"])

33 documents, 241,533 words


,qid,query,answer_span,source,expected_answer
0,Q1,Who is covered by Niramai's Code of Conduct?,is applicable to all the employees appointed on the payroll of Niramai Health Analytix Private Limited,niramai_code_of_conduct.txt,"The code applies to employees appointed on Niramai Health Analytix's payroll. It guides their behavior and attitudes during employment and provides ethical guidance for carrying out official responsibilities, rather than describing a separate policy for outside stakeholders."
1,Q2,Why did Astral establish a vigil mechanism and whistle-blower policy?,"Such vigil mechanism not only helps to detect fraud in organizations, but is also used as a corporate governance tool, which prevents and deters fraudulent activity",astral_whistle_blower_policy.txt,"Astral describes the mechanism as a means for directors and employees to raise concerns about unethical conduct, suspected or actual fraud, and code violations. It also supports corporate governance by helping detect and deter fraudulent activity."
2,Q3,How do the two laptop-support arrangements in MyGov's policy differ?,provision of a fixed monthly disbursement (“Laptop Charges”) for use of personal laptops,mygov_laptop_policy.txt,"One arrangement provides a fixed monthly payment for employees using their own laptops under the BYOD model. The other supplies an organization-owned laptop for roles needing advanced technical capabilities, with applicable provisions depending on the device category allocated."
3,Q4,How is eligibility for a laptop or laptop charges determined at MyGov?,Eligibility will be determined by the department head or immediate supervisor,mygov_laptop_policy.txt,"Eligibility depends on the employee's role, job requirements and technical needs, and is decided by the department head or immediate supervisor. The policy covers both payments for personal-laptop use and organization-owned devices, with provisions depending on the assigned category."
4,Q5,How is a verbal sexual-harassment complaint converted for formal handling at Bajaj Broking?,"In case of a verbal complaint, the complaint will be converted to a written complaint by the receiver of the complaint and consent of the complainant will be obtained",bajaj_broking_posh_policy.txt,"When formal intervention is warranted, the complaint must be put in writing. If the complaint was initially verbal, the person receiving it writes it down and obtains the complainant's consent before it enters the formal redressal process."
5,Q6,How do Bajaj Broking managers' harassment-prevention duties differ from individual employees' duties?,Responsibilities of Managers: All managers at BFSL must ensure that no employee is subject to harassment,bajaj_broking_posh_policy.txt,"Individual employees should refuse participation in harassment, support someone rejecting unwelcome behavior, and act as witnesses when a complaint is lodged. Managers additionally must ensure their teams understand zero tolerance and prevent victimization of complainants, respondents and witnesses."
6,Q7,How does JioStar's listed harassment scope distinguish verbal from visual conduct?,"b. Visual conduct such as derogatory and/or sexually oriented posters, offensive or obscene photography, cartoons, drawings or gestures",jiostar_posh_policy.txt,"Verbal examples include derogatory jokes, sexual comments and unwanted advances, whereas visual examples include offensive pictures, posters, drawings and gestures. The policy therefore covers both spoken behavior and material displayed or shown in the workplace."
7,Q8,Does JioStar's harassment scope include retaliation for taking part in an investigation?,"Retaliation for having reported or threatened to report harassment, or for opposing unlawful harassment, or for participating in an investigation",jiostar_posh_policy.txt,"Yes, the listed scope includes retaliation for reporting or threatening to report harassment, opposing unlawful harassment, or participating in an investigation. It theref

Out-of-corpus query (D1.3): Does MyGov allow employees to bring their pet dogs to the office, and what rules apply?


In [3]:
embedder = retrieval.load_embedder()
emb_tok = embedder.tokenizer
limit = embedder.max_seq_length
doc_tokens = {name: len(emb_tok(text, add_special_tokens=False, verbose=False)["input_ids"]) for name, text in docs.items()}
print(f"Embedding model: {config.EMBED_MODEL} | max input length: {limit} tokens "
      f"(incl. [CLS]/[SEP], so up to {limit - 2} content tokens per chunk)")
print(f"Corpus size with this tokenizer: {sum(doc_tokens.values()):,} tokens "
      f"(largest document {max(doc_tokens.values()):,}, smallest {min(doc_tokens.values()):,})")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model: BAAI/bge-small-en-v1.5 | max input length: 512 tokens (incl. [CLS]/[SEP], so up to 510 content tokens per chunk)
Corpus size with this tokenizer: 316,899 tokens (largest document 76,882, smallest 480)


**Why not `all-MiniLM-L6-v2`?** Its limit is 256 tokens, so 400- and 500-token chunks would be silently cut to their first 256 tokens. We therefore use **`BAAI/bge-small-en-v1.5`** (384-dim, 512-token limit, the same size class as MiniLM), so every chunk size is embedded in full. Token counts are always taken with this model's own tokenizer. Following the model card, queries get the prefix *"Represent this sentence for searching relevant passages: "*; chunks are embedded as they are.

# Part A — Dense search and chunk size (5 marks)
## A1. Build dense search (2 marks)
### A1.1 One reusable function (1 mark)
`DenseIndex` embeds a set of chunks, builds a FAISS `IndexFlatIP` and returns the top-k chunks for a query.

In [4]:
print(inspect.getsource(retrieval.DenseIndex))

class DenseIndex:
    """Embeds chunks, stores them in a FAISS IndexFlatIP and returns the top-k for a query.

    Embeddings are L2-normalised, so inner product = cosine similarity.
    """

    def __init__(self, chunks, embedder, batch_size=config.EMBED_BATCH,
                 query_instruction=config.QUERY_INSTRUCTION):
        self.chunks, self.embedder, self.query_instruction = chunks, embedder, query_instruction
        self.batch_size = batch_size
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        vectors = embedder.encode([c.text for c in chunks], batch_size=batch_size,
                                  normalize_embeddings=True, convert_to_numpy=True,
                                  show_progress_bar=False).astype("float32")
        self.index = faiss.IndexFlatIP(vectors.shape[1])
        self.index.add(vectors)
        self.build_seconds = time.perf_counter() - t0
        self.dim = vectors.shape[1]

    def 

## A2. Choose a chunk size (3 marks)
### A2.1 Fixed-size chunking at 100, 400 and 500 tokens, no overlap (2 marks)
The brief's text asks for 100 and 400 tokens, while its table lists 100 and 500, so all three sizes are run. Chunks are cut every *n* tokens of the embedding tokenizer, never across documents, and their text is taken from the original document, so a span can be matched exactly.

**Columns:**
* *Total chunks* — the absolute count (the table's "(1M)" header appears to be a typo).
* *Queries answerable* — queries whose span lies inside a single chunk; a span that straddles a chunk boundary cannot be a hit at that size.

In [5]:
size_rows, size_runs = [], {}
for size in config.CHUNK_SIZES:
    assert size + 2 <= limit, f"{size}-token chunks exceed the embedding limit"
    chunks = chunk_fixed(docs, emb_tok, size)
    index = retrieval.DenseIndex(chunks, embedder)
    results, _ = retrieval.timed_search(index, queries, k=3)
    size_runs[size] = (chunks, index, results)
    size_rows.append({"Chunk size (tokens)": size, "Total chunks": len(chunks),
                      "Hit rate@3 (%)": retrieval.hit_at(results, queries, chunks, 3),
                      "Tokens sent per query (top 3)": round(np.mean([sum(chunks[c].n_tokens for c, _ in h[:3]) for h in results]), 1),
                      "Queries answerable (span inside one chunk)": sum(bool(answer_chunk_ids(chunks, q["answer_span"])) for q in queries),
                      "Index build time (s)": round(index.build_seconds, 2)})
size_table = pd.DataFrame(size_rows)
size_table.to_csv(config.RESULTS / "a2_chunk_sizes.csv", index=False)
display(size_table)
for size, (chunks, _, results) in size_runs.items():
    print(f"{size:>3} tokens - rank of first hit per query:",
          [retrieval.first_hit_rank(h, q, chunks) for h, q in zip(results, queries)])

,Chunk size (tokens),Total chunks,Hit rate@3 (%),Tokens sent per query (top 3),Queries answerable (span inside one chunk),Index build time (s)
0,100,3183,40.0,300.0,9,2.22
1,400,810,70.0,1172.8,10,1.59
2,500,649,80.0,1457.2,10,1.57


100 tokens - rank of first hit per query: [1, 1, 3, 1, None, None, None, None, None, None]
400 tokens - rank of first hit per query: [1, 1, 2, 1, None, None, None, 1, 2, 1]
500 tokens - rank of first hit per query: [1, 1, 2, 1, None, 3, 1, None, 2, 1]


### Choosing the size (rule applied in code; `CHUNK_SIZE=<n>` overrides it)
1. Highest hit rate@3.
2. On a tie, fewer tokens sent per query.
3. Three chunks must fit the Assignment 1 model's 2,048-token context together with the prompt and answer.

In [6]:
def pick_size(table, budget=2048 - 400):   # ~400 tokens for instructions, question and answer
    ok = table[table["Tokens sent per query (top 3)"] <= budget]
    best = ok.sort_values(["Hit rate@3 (%)", "Tokens sent per query (top 3)"], ascending=[False, True]).iloc[0]
    return int(best["Chunk size (tokens)"])
CHUNK_SIZE = config.CHUNK_SIZE or pick_size(size_table)
chunks, dense, _ = size_runs[CHUNK_SIZE]
print(f"Chosen chunk size: {CHUNK_SIZE} tokens ({len(chunks)} chunks)")

Chosen chunk size: 500 tokens (649 chunks)


### A1.2 Dense-search report for the final chunk set (1 mark)
Build time = embedding all chunks plus adding them to the FAISS index.

In [7]:
a1_report = dense.report()
json.dump(a1_report, open(config.RESULTS / "a1_dense_report.json", "w"), indent=2)
display(pd.Series(a1_report).to_frame("value"))

,value
embedding model,BAAI/bge-small-en-v1.5
vector size,384
similarity,"cosine (inner product of L2-normalised vectors, FAISS IndexFlatIP)"
max input tokens,512
batch size,64
chunks indexed,649
device,cuda
index build time (s),1.57


### A2.2 Chosen chunk size and why (1 mark)
**500 tokens.** It has the highest hit rate@3: 80 %, against 70 % at 400 tokens and 40 % at 100. The other two sizes lose for different reasons:
* **100 tokens:** a clause is cut into pieces. One answer span does not fit in any single chunk, and the short chunks lack the surrounding wording that matches the questions.
* **400 tokens:** close, but a lower hit rate (70 %).

The cost of 500 is about 1,460 tokens of context per query, five times the 100-token setting. That is easy for Qwen2.5 (32 K-token context). For the 2,048-token Assignment 1 model, the full prompt (about 1,200 words) only just fits, which leaves little room for the answer.

# Part B — Keyword and hybrid search (4 marks)
All three methods use the chunks chosen in Part A and return the top 5 for each of the same 10 queries.
* **Latency:** the time to return the top 5 for one query, averaged over the 10 queries, after one untimed warm-up query.
* **Hybrid:** fuses the **top 20** of each method with RRF(d) = 1/(60 + rank in dense list) + 1/(60 + rank in BM25 list). Ranks start at 1, and a chunk missing from one list gets nothing from that list.

In [8]:
print(inspect.getsource(retrieval.rrf))
bm25 = retrieval.BM25Index(chunks)
hybrid = retrieval.HybridIndex(dense, bm25)
methods = {"Dense": dense, "BM25": bm25, "Hybrid (RRF)": hybrid}
runs, rows = {}, []
for name, index in methods.items():
    results, times = retrieval.timed_search(index, queries, k=5)
    runs[name] = results
    rows.append({"Method": name, "Average query latency (ms)": round(float(np.mean(times)), 2),
                 "Hit rate@1 (%)": retrieval.hit_at(results, queries, chunks, 1),
                 "Hit rate@3 (%)": retrieval.hit_at(results, queries, chunks, 3),
                 "Hit rate@5 (%)": retrieval.hit_at(results, queries, chunks, 5)})
search_table = pd.DataFrame(rows)
search_table.to_csv(config.RESULTS / "b_search_comparison.csv", index=False)
display(search_table)
ranks = pd.DataFrame({"qid": [q["qid"] for q in queries], "query": [q["query"] for q in queries],
                      **{f"{m}: rank of first hit": [retrieval.first_hit_rank(h, q, chunks) for h, q in zip(runs[m], queries)] for m in runs}})
display(ranks)

def rrf(dense_hits, bm25_hits, k=config.RRF_K, top=config.TOP_K):
    """Reciprocal Rank Fusion: score(d) = 1/(k + rank_dense) + 1/(k + rank_bm25), ranks start at 1;
    a chunk missing from one list gets nothing from that list."""
    scores = {}
    for hits in (dense_hits, bm25_hits):
        for rank, (cid, _) in enumerate(hits, start=1):
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (k + rank)
    fused = sorted(scores.items(), key=lambda kv: -kv[1])
    return fused[:top]



,Method,Average query latency (ms),Hit rate@1 (%),Hit rate@3 (%),Hit rate@5 (%)
0,Dense,8.49,50.0,80.0,90.0
1,BM25,1.46,60.0,80.0,90.0
2,Hybrid (RRF),10.33,50.0,80.0,90.0


,qid,query,Dense: rank of first hit,BM25: rank of first hit,Hybrid (RRF): rank of first hit
0,Q1,Who is covered by Niramai's Code of Conduct?,1.0,1.0,1.0
1,Q2,Why did Astral establish a vigil mechanism and whistle-blower policy?,1.0,1.0,1.0
2,Q3,How do the two laptop-support arrangements in MyGov's policy differ?,2.0,2.0,2.0
3,Q4,How is eligibility for a laptop or laptop charges determined at MyGov?,1.0,1.0,1.0
4,Q5,How is a verbal sexual-harassment complaint converted for formal handling at Bajaj Broking?,5.0,1.0,2.0
5,Q6,How do Bajaj Broking managers' harassment-prevention duties differ from individual employees' duties?,3.0,NaN,NaN
6,Q7,How does JioStar's listed harassment scope distinguish verbal from visual conduct?,1.0,1.0,1.0
7,Q8,Does JioStar's harassment scope include retaliation for taking part in an investigation?,NaN,4.0,4.0
8,Q9,What initial action does Aurobindo describe after receiving a human-rights complaint?,2.0,2.0,2.0
9,Q10,How does Aurobindo distinguish the preliminary inquiry from its detailed grievance inquiry?,1.0,1.0,1.0


### Best method (rule applied in code; `BEST_METHOD=<name>` overrides it)
The best method has the highest hit rate@3; ties are broken by hit rate@1, then by lower latency.

In [9]:
best_row = search_table.sort_values(["Hit rate@3 (%)", "Hit rate@1 (%)", "Average query latency (ms)"],
                                    ascending=[False, False, True]).iloc[0]
BEST = os.environ.get("BEST_METHOD") or best_row["Method"]
print("Best method for Part C:", BEST)

Best method for Part C: BM25


### B4. Which method works best and why, with one example query (1 mark)
**BM25.** All three methods put the answer span in the top 3 for 8 of the 10 queries and in the top 5 for 9. BM25 wins on two counts:
* It ranks the span **first** more often: hit rate@1 is 60 %, against 50 % for dense and hybrid.
* It is about 6× faster: 1.4 ms per query, against 8.8 ms for dense and 10.4 ms for hybrid.

The questions reuse the policies' own wording and organisation names, which exact-term matching rewards.

**Example — Q5,** "How is a *verbal* sexual-harassment *complaint converted* for formal handling at *Bajaj Broking*?" BM25 ranks the answer chunk 1st because it shares exactly those words ("In case of a verbal complaint, the complaint will be converted …"); dense search ranks it 5th, below other Bajaj POSH chunks with similar meaning.

**Dense still has strengths.** Q6 (managers' duties) is found only by dense search (rank 3). With equal hit@3, RRF did not combine the two strengths here: hybrid matched dense on hit@1 and was the slowest.

# Part C — Reranking (4 marks)
### C1. Rerank (1 mark)
The best Part B method returns the top 10 for each query. A cross-encoder, `cross-encoder/ms-marco-MiniLM-L-6-v2`, re-scores each (query, chunk) pair and the top 3 are kept. Reranking time per query covers scoring the 10 pairs and sorting them, averaged after one warm-up query.

In [10]:
reranker = rr.load_reranker()
before10, _ = retrieval.timed_search(methods[BEST], queries, k=10)
after3, rerank_ms = rr.timed_rerank(reranker, queries, before10, chunks, keep=3)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

### C2. Top 3 before and after reranking for each query (1 mark)
`[chunk id] source`; ✓ marks a chunk that contains the answer span.

In [11]:
def show(hits, q):
    return " | ".join(f"[{c}] {chunks[c].source}{' ✓' if is_hit(chunks[c].text, q['answer_span']) else ''}" for c, _ in hits[:3])
compare = pd.DataFrame([{"qid": q["qid"], "query": q["query"], f"top 3 before ({BEST})": show(b, q),
                         "top 3 after reranking": show(a, q), "first result changed": b[0][0] != a[0][0]}
                        for q, b, a in zip(queries, before10, after3)])
compare.to_csv(config.RESULTS / "c_rerank_comparison.csv", index=False)
display(compare)

,qid,query,top 3 before (BM25),top 3 after reranking,first result changed
0,Q1,Who is covered by Niramai's Code of Conduct?,[636] niramai_code_of_conduct.txt ✓ | [642] niramai_code_of_conduct.txt | [66] beml_whistle_blower_policy.txt,[636] niramai_code_of_conduct.txt ✓ | [87] ccs_conduct_rules_1964.txt | [407] indus_university_hr_manual.txt,False
1,Q2,Why did Astral establish a vigil mechanism and whistle-blower policy?,[32] astral_whistle_blower_policy.txt ✓ | [64] beml_whistle_blower_policy.txt | [206] growxcd_vigil_mechanism_policy.txt,[32] astral_whistle_blower_policy.txt ✓ | [524] khannapaper_vigil_mechanism_policy.txt | [64] beml_whistle_blower_policy.txt,False
2,Q3,How do the two laptop-support arrangements in MyGov's policy differ?,[629] mygov_laptop_policy.txt | [627] mygov_laptop_policy.txt ✓ | [625] mygov_internship_program.txt,[632] mygov_talent_development_policy.txt | [627] mygov_laptop_policy.txt ✓ | [633] mygov_talent_development_policy.txt,True
3,Q4,How is eligibility for a laptop or laptop charges determined at MyGov?,[627] mygov_laptop_policy.txt ✓ | [629] mygov_laptop_policy.txt | [628] mygov_laptop_policy.txt,[627] mygov_laptop_policy.txt ✓ | [628] mygov_laptop_policy.txt | [629] mygov_laptop_policy.txt,False
4,Q5,How is a verbal sexual-harassment complaint converted for formal handling at Bajaj Broking?,[51] bajaj_broking_posh_policy.txt ✓ | [46] bajaj_broking_posh_policy.txt | [47] bajaj_broking_posh_policy.txt,[47] bajaj_broking_posh_policy.txt | [46] bajaj_broking_posh_policy.txt | [51] bajaj_broking_posh_policy.txt ✓,True
5,Q6,How do Bajaj Broking managers' harassment-prevention duties differ from individual employees' duties?,[47] bajaj_broking_posh_policy.txt | [46] bajaj_broking_posh_policy.txt | [90] ccs_conduct_rules_1964.txt,[47] bajaj_broking_posh_policy.txt | [46] bajaj_broking_posh_policy.txt | [268] iim_raipur_hr_policy_service_rules.txt,False
6,Q7,How does JioStar's listed harassment scope distinguish verbal from visual conduct?,[522] jiostar_posh_policy.txt ✓ | [517] jiostar_posh_policy.txt | [48] bajaj_broking_posh_policy.txt,[522] jiostar_posh_policy.txt ✓ | [49] bajaj_broking_posh_policy.txt | [76] bptp_posh_policy.txt,False
7,Q8,Does JioStar's harassment scope include retaliation for taking part in an investigation?,[522] jiostar_posh_policy.txt | [5] 3m_posh_policy.txt | [0] 3m_posh_policy.txt,[522] jiostar_posh_policy.txt | [204] ghl_vigil_mechanism_policy.txt | [523] jiostar_posh_policy.txt ✓,False
8,Q9,What initial action does Aurobindo describe after receiving a human-rights complaint?,[42] aurobindo_human_rights_policy.txt | [44] aurobindo_human_rights_policy.txt ✓ | [43] aurobindo_human_rights_policy.txt,[44] aurobindo_human_rights_policy.txt ✓ | [42] aurobindo_human_rights_policy.txt | [58] bajaj_broking_posh_policy.txt,True
9,Q10,How does Aurobindo distinguish the preliminary inquiry from its detailed grievance inquiry?,[44] aurobindo_human_rights_policy.txt ✓ | [58] bajaj_broking_posh_policy.txt | [271] iim_raipur_hr_policy_service_rules.txt,[44] aurobindo_human_rights_policy.txt ✓ | [43] aurobindo_human_rights_policy.txt | [42] aurobindo_human_rights_policy.txt,False


### C3. Reranking measures (1 mark)
*Rank change rate* is the share of the 10 queries whose first result changes after reranking.

In [12]:
rerank_table = pd.DataFrame({"Measure": ["Reranking time per query (ms)", "Rank change rate (%)",
                                         f"Hit rate@1 before reranking (from Part B, {BEST})", "Hit rate@1 after reranking",
                                         f"Hit rate@3 before reranking ({BEST})", "Hit rate@3 after reranking"],
                             "Value": [round(float(np.mean(rerank_ms)), 1), 100 * compare["first result changed"].mean(),
                                       retrieval.hit_at(before10, queries, chunks, 1), retrieval.hit_at(after3, queries, chunks, 1),
                                       retrieval.hit_at(before10, queries, chunks, 3), retrieval.hit_at(after3, queries, chunks, 3)]})
rerank_table.to_csv(config.RESULTS / "c_rerank_measures.csv", index=False)
display(rerank_table)

,Measure,Value
0,Reranking time per query (ms),22.3
1,Rank change rate (%),30.0
2,"Hit rate@1 before reranking (from Part B, BM25)",60.0
3,Hit rate@1 after reranking,60.0
4,Hit rate@3 before reranking (BM25),80.0
5,Hit rate@3 after reranking,90.0


### C4. Did reranking help? (1 mark)
**Not for the first result.** Hit rate@1 is 60 % both before and after reranking. The cross-encoder changed the top result for 3 of 10 queries (rank change rate 30 %):
* **One gain:** Q9's answer chunk moved from 2nd to 1st.
* **Two losses:**
  * Q5's answer chunk dropped from 1st to 3rd.
  * In Q3, a Talent-Development-policy chunk moved above the laptop-policy chunks.

**It did help the context the generator reads.** Hit rate@3 rose from 80 % to 90 %, because Q8's answer chunk moved from rank 4 into the top 3.

**Cost:** 22.9 ms per query, about 16× the BM25 search itself, but negligible next to answer generation (seconds per query). It is worth keeping for the top-3 context and not worth it for top-1 accuracy alone.

# Part D — Generate and check answers (7 marks)
## D1. Generate grounded answers (3 marks)
### D1.1 The prompt (1 mark)
Each query gets the **top 3 chunks after reranking**, numbered [1]–[3]. Decoding is greedy (temperature 0).

In [13]:
print("SYSTEM:", gen.GEN_SYSTEM, "\n")
print(gen.GEN_TEMPLATE)

SYSTEM: You are an HR-policy assistant. You answer questions strictly from the document chunks you are given and never use outside knowledge. 

Answer the question using ONLY the numbered chunks below.

Rules:
1. Use only facts stated in the chunks. Do not add facts from outside knowledge.
2. Cite the chunk number(s) you used in square brackets after each sentence, for example [2].
3. If the chunks do not contain the answer, reply exactly: Not found in the documents.
4. Answer in 1 to 3 sentences.

Chunks:
{chunks}

Question: {question}
Answer:


In [14]:
# retrieval for the out-of-corpus query uses the same pipeline (best method top 10 -> rerank -> top 3)
ooc_q = {"qid": ooc["qid"], "query": ooc["query"], "expected_answer": ooc["expected_answer"], "answer_span": ""}
ooc_after3 = rr.rerank(reranker, ooc_q["query"], methods[BEST].search(ooc_q["query"], 10), chunks, keep=3)
all_q, all_ctx = queries + [ooc_q], after3 + [ooc_after3]
contexts = [[chunks[c] for c, _ in hits] for hits in all_ctx]
prompts = [gen.gen_prompt(q["query"], ctx) for q, ctx in zip(all_q, contexts)]
del embedder, reranker; gen.free()
print(f"Prompt length for Q1 with {CHUNK_SIZE}-token chunks: {len(prompts[0].split())} words")

Prompt length for Q1 with 500-token chunks: 1200 words


### First try: the Assignment 1 model (TinyLlama-1.1B CPT + Adapter B)
The brief allows switching to an instruct model if the Assignment 1 model ignores the chunks or never says "Not found". The same prompts are run through it first, and three behaviours are checked automatically: whether it cites a chunk, whether it says "Not found", and whether the answer stops cleanly.

In [15]:
a1_rows = []
if (config.A1_BASE / "config.json").exists() and (config.A1_ADAPTER / "adapter_config.json").exists():
    a1_model, a1_tok = gen.load_a1_model()
    for q, p in zip(all_q, prompts):
        ans = gen.chat(a1_model, a1_tok, gen.GEN_SYSTEM, p)
        a1_rows.append({"qid": q["qid"], "query": q["query"], "A1 model answer": ans,
                        "cites a chunk": bool(gen.cites(ans)), "says Not found": gen.says_not_found(ans)})
    del a1_model; gen.free()
    a1_answers = pd.DataFrame(a1_rows)
    a1_answers.to_csv(config.RESULTS / "d1_a1_model_answers.csv", index=False)
    display(a1_answers)
    print(f"A1 model: cites a chunk in {a1_answers['cites a chunk'].sum()}/11 answers; "
          f"says 'Not found' for the out-of-corpus query: {bool(a1_answers['says Not found'].iloc[-1])}")
elif (config.RESULTS / "d1_a1_model_answers.csv").exists():   # model removed from the lab disk after an earlier run
    a1_answers = pd.read_csv(config.RESULTS / "d1_a1_model_answers.csv")
    print(f"Assignment 1 model files are no longer under {config.A1_PROJECT}; showing its saved answers from the earlier run "
          "(same prompts, greedy decoding).")
    display(a1_answers)
    print(f"A1 model: cites a chunk in {a1_answers['cites a chunk'].sum()}/11 answers; "
          f"says 'Not found' for the out-of-corpus query: {bool(a1_answers['says Not found'].iloc[-1])}")
else:
    print(f"Assignment 1 model not found under {config.A1_PROJECT} - set A1_PROJECT to the CorpPolicyLM folder.")

Assignment 1 model files are no longer under /home/jovyan/datavol-1/CorpPolicyLM; showing its saved answers from the earlier run (same prompts, greedy decoding).


,qid,query,A1 model answer,cites a chunk,says Not found
0,Q1,Who is covered by Niramai's Code of Conduct?,"The Code of Conduct applies to all employees, including temporary, part-time, and casual staff. It also applies to contractors and consultants.",False,False
1,Q2,Why did Astral establish a vigil mechanism and whistle-blower policy?,"The mechanism is intended tob, and the whistle-purchase|\n\nThe mechanism, the whistle-purchase and\n\nThe mechanism, and the Company policy\nThe whistle purchase\n\nThe mechanism\n\nThe Company\n\nThe purchase\n\nThe mechanism\n\n\n\n\n\n\n\n\n\n\n\n\nThe purchase\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nor and and and and and ors or or or or or or or or",False,False
2,Q3,How do the two laptop-support arrangements in MyGov's policy differ?,"The organization provides a fixed monthly payment for the use of a laptop, but the employee is expected to cover the cost of the laptop's purchase and maintenance. The policy also provides for reimbursement of the cost of the laptop's purchase and maintenance, but the employee is expected to cover the cost of the laptop's purchase and maintenance.",False,False
3,Q4,How is eligibility for a laptop or laptop charges determined at MyGov?,The eligibility for laptop charges is based on the employee’s role and responsibilities. The eligibility for laptop charges is also based on the employee’s role and responsibilities. The eligibility for laptop charges is also based on the employee’s role and responsibilities.,False,False
4,Q5,How is a verbal sexual-harassment complaint converted for formal handling at Bajaj Broking?,"The IC will receive the complaint and refer it to the IC for the workplace, which will manage the redressal process. The IC wills the process, which is sexual harassment at work, and handle the redressal, which is the IC, which the work and manage the redressal, and the IC.\n\nThe IC, which work and handle the redness.",False,False
5,Q6,How do Bajaj Broking managers' harassment-prevention duties differ from individual employees' duties?,"The company's policy is to prevent, and take disciplinary action against the perpetrator, and to provide for the aggrieved person's name, address, telephone number, and email address, and to provide the policy's provisions, and the company's policy, and disciplinary action, and providing the aggrants, and providing, and email, and email, and sms, and email, and providing, and providing, and providing, and providing, and providing, and providing ands, and providing, andsafact or providing, and providing, and providing and providing ands.\n\nand providing, providing, and providing ands\nand providing and\n\nand\nands\n\n\nand and andand ands ands ands ands ands ands ands andand ands ands ands and and ands ands ands ands ands ands ands ands and and and",False,False
6,Q7,How does JioStar's listed harassment scope distinguish verbal from visual conduct?,"The scope of the policy includes verbal conduct such as sexually suggestive or offensive remarks, jokes, inappropriate comments about a person's clothing or body, and physical conduct such as unwanted physical contact or physical assault. It also covers unwelcome physical contact such as grabbing, pinching, or pushing.",False,False
7,Q8,Does JioStar's harassment scope include retaliation for taking part in an investigation?,"The policy covers sexual harassment, unwelcome sexually determined conduct, and retaliation against individuals who report or participate in an investigation. The policy also prohibits retaliation against witnesses and provides for disciplinary action against perpetrators.",False,False
8,Q9,What initial action does Aurobindo describe after receiving a human-rights complaint?,"The company takes the complaint seriously and initiates an inquiry. The inquiry is completed and the findings are reported

A1 model: cites a chunk in 0/11 answers; says 'Not found' for the out-of-corpus query: False


**Problem observed with the Assignment 1 model, and the switch.** The Assignment 1 model (TinyLlama-1.1B CPT + Adapter B) ignored both the chunks and the instructions:
* It cited a chunk in **0 of 11** answers and never replied "Not found".
* For the out-of-corpus question it invented a policy: *"The company is a pet-friendly workplace, and employees are free to bring their pets to the office. However, the company requires that the animal is kept on a leash …"*
* Other answers contradict their chunks. In Q1 it says Niramai's code covers temporary, part-time and casual staff and contractors; chunk [1] says employees on the payroll.
* Some answers degenerate into repetition (Q2, Q6).

As the brief allows, generation therefore switches to the instruct model **Qwen2.5-3B-Instruct** (4-bit), with the same prompt and the same top-3 chunks.

### D1.2 Answers of the generator (1 mark) and D1.3 the out-of-corpus query (1 mark)

In [16]:
print(f"Generator: {config.GENERATOR_NAME}  [{config.GENERATOR}]  - free disk before download: {gen.free_gb():.1f} GiB")
gen.make_room(config.GENERATOR, drop=config.JUDGE)   # re-runs: the judge may still be cached
g_model, g_tok = gen.load_instruct(config.GENERATOR)
answers = [gen.chat(g_model, g_tok, gen.GEN_SYSTEM, p) for p in prompts]
del g_model; gen.free()
gen_table = pd.DataFrame([{"qid": q["qid"], "query": q["query"], "answer": a, "cited chunks": gen.cites(a),
                           "chunks used (source)": " | ".join(f"[{i}] {c.source}" for i, c in enumerate(ctx, 1))}
                          for q, a, ctx in zip(all_q, answers, contexts)])
gen_table.to_csv(config.RESULTS / "d1_generator_answers.csv", index=False)
json.dump([{"qid": q["qid"], "query": q["query"], "expected_answer": q["expected_answer"], "answer": a,
            "chunk_ids": [c for c, _ in h]} for q, a, h in zip(all_q, answers, all_ctx)],
          open(config.RESULTS / "d1_answers.json", "w"), indent=2, ensure_ascii=False)
display(gen_table.iloc[:10])
print("D1.3 - out-of-corpus query:", ooc_q["query"])
print("Answer:", answers[-1])
print("Says 'Not found in the documents':", gen.says_not_found(answers[-1]))

Generator: Qwen2.5-3B-Instruct (4-bit NF4)  [unsloth/Qwen2.5-3B-Instruct-bnb-4bit]  - free disk before download: 0.6 GiB


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


,qid,query,answer,cited chunks,chunks used (source)
0,Q1,Who is covered by Niramai's Code of Conduct?,The Code of Conduct applies to all the employees appointed on the payroll of Niramai Health Analytix Private Limited. [1],[1],[1] niramai_code_of_conduct.txt | [2] ccs_conduct_rules_1964.txt | [3] indus_university_hr_manual.txt
1,Q2,Why did Astral establish a vigil mechanism and whistle-blower policy?,"Astral Limited established a vigil mechanism and whistle-blower policy because it is mandated by the Companies Act, 2013 read with rules made thereunder and regulation 22 of SEBI (Listing Obligations and Disclosure requirements) regulations, 2015, as stated in [1]. Additionally, it aligns with the company's belief in conducting its affairs in a fair and transparent manner by adopting the highest standards of professionalism, honesty, integrity, and ethical behavior, as mentioned in [1].",[1],[1] astral_whistle_blower_policy.txt | [2] khannapaper_vigil_mechanism_policy.txt | [3] beml_whistle_blower_policy.txt
2,Q3,How do the two laptop-support arrangements in MyGov's policy differ?,Not found in the documents.,[],[1] mygov_talent_development_policy.txt | [2] mygov_laptop_policy.txt | [3] mygov_talent_development_policy.txt
3,Q4,How is eligibility for a laptop or laptop charges determined at MyGov?,"Eligibility for a laptop or laptop charges at MyGov is determined by the department head or immediate supervisor based on the employee's role, job requirements, and technical needs. [1]",[1],[1] mygov_laptop_policy.txt | [2] mygov_laptop_policy.txt | [3] mygov_laptop_policy.txt
4,Q5,How is a verbal sexual-harassment complaint converted for formal handling at Bajaj Broking?,"A verbal sexual-harassment complaint is converted to a written complaint when it is received by the ethics and compliance team. Additionally, the receiver of the complaint must obtain the consent of the complainant for this conversion [3].",[3],[1] bajaj_broking_posh_policy.txt | [2] bajaj_broking_posh_policy.txt | [3] bajaj_broking_posh_policy.txt
5,Q6,How do Bajaj Broking managers' harassment-prevention duties differ from individual employees' duties?,Not found in the documents.,[],[1] bajaj_broking_posh_policy.txt | [2] bajaj_broking_posh_policy.txt | [3] iim_raipur_hr_policy_service_rules.txt
6,Q7,How does JioStar's listed harassment scope distinguish verbal from visual conduct?,"JioStar's scope distinguishes verbal from visual conduct by listing specific examples for each category. For verbal conduct, it includes ""epithets, derogatory jokes or comments, slurs or unwanted sexual advances, invitations or comments, questions about a person’s sexual practices, or gossiping about sexual relations."" For visual conduct, it specifies ""derogatory and/or sexually oriented posters, offensive or obscene photography, cartoons, drawings or gestures, display of sexually suggestive or lewd objects."" [1]",[1],[1] jiostar_posh_policy.txt | [2] bajaj_broking_posh_policy.txt | [3] bptp_posh_policy.txt
7,Q8,Does JioStar's harassment scope include retaliation for taking part in an investigation?,Not found in the documents.,[],[1] jiostar_posh_policy.txt | [2] ghl_vigil_mechanism_policy.txt | [3] jiostar_posh_policy.txt
8,Q9,What initial action does Aurobindo describe after receiving a human-rights complaint?,"Aurobindo describes that after receiving a human-rights complaint, the preliminary enquiry will be conducted by involving the complainant in a way that the matter will be dealt with utmost confidentiality within 3 days time. [5.2]",[],[1] aurobindo_human_rights_policy.txt | [2] aurobindo_human_rights_policy.txt | [3] bajaj_broking_posh_policy.txt
9,Q10,How does Aurobindo distinguish the preliminary inquiry from its detailed grievance inquiry?,"The preliminary enquiry involves initial contact with the complainant to understand the nature of the issue and determine if there is a prima facie case. It is conducted by involving the complainant in a confidential manner within 3 da

D1.3 - out-of-corpus query: Does MyGov allow employees to bring their pet dogs to the office, and what rules apply?
Answer: Not found in the documents.
Says 'Not found in the documents': True


## D2. Check the answers with an LLM judge (4 marks)
### D2.1 Judge prompt and results (1 mark)
**Judge model:** Llama (printed below), a different model family from the Qwen generator. Decoding is greedy, so it behaves like temperature 0 and gives the same verdict every time.

The judge is given the question, the expected answer from the Assignment 1 evaluation set, the three retrieved chunks and the generator's answer. It returns four fixed lines (`CORRECT`, `SUPPORTED`, `REASON`, `UNSUPPORTED`), which are parsed with code.

**Constrained decoding of the format.** On a first run, the 3B judge left the CORRECT and SUPPORTED fields empty in all 10 replies. The reply is therefore built line by line from the same prompt:
* **Labels:** for `CORRECT` and `SUPPORTED`, the judge chooses among the allowed labels only, taking the label whose first token it rates most probable. This is argmax, i.e. temperature 0.
* **Explanations:** `REASON` and `UNSUPPORTED` are then generated greedily, conditioned on the chosen labels.

In [17]:
print("SYSTEM:", gen.JUDGE_SYSTEM, "\n")
print(gen.JUDGE_TEMPLATE)

SYSTEM: You are a strict examiner who grades answers from an HR-policy question-answering system. 

Grade the SYSTEM ANSWER below. Check two things separately.

CHECK 1 - CORRECT: compare the system answer with the EXPECTED ANSWER.
- correct: it states the main point(s) of the expected answer and contradicts nothing in it.
- partly correct: it states some of the expected answer's points but misses an important one or adds an error, without contradicting the main point.
- wrong: it misses or contradicts the main point, answers a different question, or is "Not found in the documents" when the expected answer exists.
If the expected answer is itself "Not found in the documents", then a "Not found" reply is correct and any invented answer is wrong.
Ignore citation numbers and wording; judge the meaning.

CHECK 2 - SUPPORTED: compare every statement in the system answer with the RETRIEVED CHUNKS only.
- yes: every statement can be found in (or directly follows from) the chunks.
- no: at lea

In [18]:
gen.make_room(config.JUDGE, drop=config.GENERATOR)   # small disk: one model at a time (answers are already saved)
print(f"Judge: {config.JUDGE_NAME}  [{config.JUDGE}]")
j_model, j_tok = gen.load_instruct(config.JUDGE)
judged = [gen.judge(j_model, j_tok, gen.judge_prompt(q["query"], q["expected_answer"], ctx, a))
          for q, a, ctx in zip(queries, answers[:10], contexts[:10])]
del j_model; gen.free()
verdicts = pd.DataFrame([{"qid": q["qid"], "query": q["query"], **v} for q, (v, _) in zip(queries, judged)])
raw = [r for _, r in judged]
verdicts["raw judge output"] = raw
verdicts.to_csv(config.RESULTS / "d2_judge_verdicts.csv", index=False)
display(verdicts[["qid", "query", "correct", "supported", "reason", "unsupported quote"]])
summary = pd.DataFrame({"Check": ["Correct", "Partly correct", "Wrong", "Supported by the chunks"],
                        "Count out of 10": [int((verdicts.correct == "correct").sum()), int((verdicts.correct == "partly correct").sum()),
                                            int((verdicts.correct == "wrong").sum()), int((verdicts.supported == "yes").sum())]})
display(summary)
print("Unparsed verdicts:", int((verdicts[["correct", "supported"]] == "unparsed").any(axis=1).sum()))

Judge: Llama-3.2-3B-Instruct (4-bit NF4)  [unsloth/Llama-3.2-3B-Instruct-bnb-4bit]


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[transformers] Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[transformers] Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


,qid,query,correct,supported,reason,unsupported quote
0,Q1,Who is covered by Niramai's Code of Conduct?,correct,yes,The system answer states the main point of the expected answer and is supported by the retrieved chunk [1].,None
1,Q2,Why did Astral establish a vigil mechanism and whistle-blower policy?,partly correct,yes,"The system answer states some of the expected answer's points but misses an important one, specifically the objective of the policy, which is to extract maximum information on activities affecting the Company, its values, philosophy, principles, and beliefs, including violations of legal or regulatory requirements, incorrect or misrepresentation of financial statements, reports, etc.","""As stated in [1]"""
2,Q3,How do the two laptop-support arrangements in MyGov's policy differ?,wrong,yes,The system answer does not contradict the main point of the expected answer and can be found in the retrieved chunks.,None
3,Q4,How is eligibility for a laptop or laptop charges determined at MyGov?,correct,yes,The system answer accurately states the main point of the expected answer and is supported by the retrieved chunk [1].,None
4,Q5,How is a verbal sexual-harassment complaint converted for formal handling at Bajaj Broking?,correct,yes,The system answer states the main point of the expected answer and is supported by the retrieved chunk [3].,None
5,Q6,How do Bajaj Broking managers' harassment-prevention duties differ from individual employees' duties?,wrong,yes,"The system answer does not contradict the main point of the expected answer, but it is missing the main point.",None
6,Q7,How does JioStar's listed harassment scope distinguish verbal from visual conduct?,correct,yes,The system answer correctly states the main point of the expected answer and is supported by the retrieved chunks.,None
7,Q8,Does JioStar's harassment scope include retaliation for taking part in an investigation?,partly correct,yes,"The system answer states that retaliation for taking part in an investigation is included in the scope, but misses the important point that it also covers adverse responses to raising concerns.","""f. Retaliation for having reported or threatened to report harassment, or for opposing unlawful harassment, or for participating in an investigation;"""
8,Q9,What initial action does Aurobindo describe after receiving a human-rights complaint?,correct,yes,The system answer states the main point of the expected answer and is supported by the retrieved chunk [5.2] which describes the preliminary enquiry process.,None
9,Q10,How does Aurobindo distinguish the preliminary inquiry from its detailed grievance inquiry?,correct,yes,"The system answer states the main point(s) of the expected answer and contradicts nothing in it, and every statement can be found in (or directly follows from) the retrieved chunks.",None


,Check,Count out of 10
0,Correct,6
1,Partly correct,2
2,Wrong,2
3,Supported by the chunks,10


Unparsed verdicts: 0


### D2.2 Spot-check of the judge (1 mark)
I read the full answers, expected answers and chunks of three queries, Q2, Q5 and Q8. Each has two verdicts, six in total, and I agree with **3 of the 6**.

| Query | Judge: correct / supported | My verdict | Agree? |
|---|---|---|---|
| Q2 Astral vigil mechanism | partly correct / yes | partly correct / yes | 2 of 2 |
| Q5 Bajaj verbal complaint | correct / yes | **partly correct / no** | 0 of 2 |
| Q8 JioStar retaliation | partly correct / yes | **wrong** / yes | 1 of 2 |

* **Q2:** The answer gives the legal mandate (Companies Act, SEBI regulation 22) and the commitment to fair, transparent conduct, both stated in chunk [1]. It misses the expected main point that the mechanism helps detect and deter fraud, so *partly correct* and *supported* are right. The judge's REASON, however, names an "objective … to extract maximum information" that is not part of the expected answer: the right verdict for a partly wrong reason.
* **Q5:** The answer says the verbal complaint is converted "when it is received by the ethics and compliance team". Chunk [3] says it is converted *by the receiver of the complaint*, with the complainant's consent. The ethics and compliance team appears in a different clause (the reporting route in 6.3). So the answer is *partly correct*, and that statement is *not supported*.
* **Disagreement explained, Q8:** The generator answered "Not found in the documents", while the expected answer exists and the span is in chunk [3]. By the judge prompt's own definition this is **wrong**. The judge chose *partly correct*, and its REASON describes an answer that was never given ("states that retaliation for taking part in an investigation is included"). It graded the expected answer and the chunks instead of the system answer. *Supported = yes* is right, because a "Not found" reply makes no claims.

**What the spot-check shows about the 3B judge.** It never answered *no* to SUPPORTED: 10 of 10 answers were marked supported. Its explanations sometimes do not match the answer it was given. **Corrected counts** from my reading of all 10 answers:
* Correct 5 (Q1, Q4, Q7, Q9, Q10)
* Partly correct 2 (Q2, Q5)
* Wrong 3 (Q3, Q6, Q8)
* Supported 9 of 10 (Q5 is not)

The judge's counts were 6 / 2 / 2 / 10. A larger judge (Llama-3.1-8B, which `JUDGE=` selects when there is enough disk) would be the first improvement.

### D2.3 Find the cause (1 mark)
The cell below shows the three chunks of every answer the judge marked wrong, partly correct or unsupported, and states whether the answer span was among them. If the span was there, generation caused the problem; if not, retrieval did.

In [19]:
flagged = verdicts[(verdicts.correct != "correct") | (verdicts.supported != "yes")]
for _, v in flagged.iterrows():
    i = next(k for k, q in enumerate(queries) if q["qid"] == v.qid)
    span_in = any(is_hit(c.text, queries[i]["answer_span"]) for c in contexts[i])
    display(Markdown(f"**{v.qid} — {queries[i]['query']}**  \nJudge: {v.correct}, supported = {v.supported}  \n"
                     f"Answer: {answers[i]}  \nExpected: {queries[i]['expected_answer']}  \n"
                     f"Answer span in the 3 chunks: **{span_in}** → likely cause: **{'generation' if span_in else 'retrieval'}**"))
    for n, c in enumerate(contexts[i], 1):
        print(f"[{n}] ({c.source}) {' '.join(c.text.split())}\n")
if flagged.empty:
    print("No answer was marked wrong, partly correct or unsupported.")

**Q2 — Why did Astral establish a vigil mechanism and whistle-blower policy?**  
Judge: partly correct, supported = yes  
Answer: Astral Limited established a vigil mechanism and whistle-blower policy because it is mandated by the Companies Act, 2013 read with rules made thereunder and regulation 22 of SEBI (Listing Obligations and Disclosure requirements) regulations, 2015, as stated in [1]. Additionally, it aligns with the company's belief in conducting its affairs in a fair and transparent manner by adopting the highest standards of professionalism, honesty, integrity, and ethical behavior, as mentioned in [1].  
Expected: Astral describes the mechanism as a means for directors and employees to raise concerns about unethical conduct, suspected or actual fraud, and code violations. It also supports corporate governance by helping detect and deter fraudulent activity.  
Answer span in the 3 chunks: **True** → likely cause: **generation**

[1] (astral_whistle_blower_policy.txt) Page | 1 Vigil Mechanism & Whistle Blower Policy Page | 2 PREAMBLE: ❖ Astral Limited (“ASTRAL” or “Company”) believes in the conduct of the affairs of its business and operations in a fair and transparent manner by adopting highest standards of professionalism, honesty, integrity and ethical behavior. ❖ The Companies Act, 2013 read with rules made thereunder and regulation 22 of SEBI (Listing Obligations and Disclosure requirements) regulations, 2015, mandates that every Listed Company to establish a mechanism for employees/directors to report concerns about unethical behavior, actual or suspected fraud or violation of the company’s code of conduct or ethics policy. ❖ Such vigil mechanism not only helps to detect fraud in organizations, but is also used as a corporate governance tool, which prevents and deters fraudulent activity. ❖ In view of the above, Astral Limited, being a listed company proposes to establish a Vigil Mechanism and to formulat

**Q3 — How do the two laptop-support arrangements in MyGov's policy differ?**  
Judge: wrong, supported = yes  
Answer: Not found in the documents.  
Expected: One arrangement provides a fixed monthly payment for employees using their own laptops under the BYOD model. The other supplies an organization-owned laptop for roles needing advanced technical capabilities, with applicable provisions depending on the device category allocated.  
Answer span in the 3 chunks: **True** → likely cause: **generation**

[1] (mygov_talent_development_policy.txt) MyGov TALENT DEVELOPMENT POLICY 1. Objectives MyGov recognizes the importance of effective learning and development for nurturing talent so that MyGov can achieve its objective of citizen engagement and participative governance. Team members of MyGov will have access to opportunities for learning and development to enable them to perform to the best of their ability, in order to help MyGov to achieve its objectives in both the short and longer term. 2. Scope The policy is inclusive and covers all team members of MyGov – whether on contract through DIC and Deputation excluding part-time, third -party hires, invoiced consultants. 3. Eligibility A team member will be eligible for a training program/course/certification after successful completion of 8 months with the organization. The Program /course/certification has to be relevant to his/her area of work, which has to be approved by the reporting Function Head and CEO, MyGov 4. Criteria for Lear

**Q6 — How do Bajaj Broking managers' harassment-prevention duties differ from individual employees' duties?**  
Judge: wrong, supported = yes  
Answer: Not found in the documents.  
Expected: Individual employees should refuse participation in harassment, support someone rejecting unwelcome behavior, and act as witnesses when a complaint is lodged. Managers additionally must ensure their teams understand zero tolerance and prevent victimization of complainants, respondents and witnesses.  
Answer span in the 3 chunks: **False** → likely cause: **retrieval**

[1] (bajaj_broking_posh_policy.txt) The monthly reporting grid: 4 1. Purpose & Background Bajaj Financial Securities Limited (“BFSL”, “Bajaj Broking”, “Company”) is committed to creating a healthy working environment that enables employees to work without fear of prejudice, gender discrimination and harassment. We believe that all employees hav e the right to be treated with fairness and dignity. We recruit competent and motivated people who respect our values. We provide equal opportunities for their development and advancement; protect their privacy and do not tolerate any form of harassment or discrimination. Sexual harassment at the workplace or any other place than workplace , is a grave offence and is punishable under law. This policy has been framed in accordance with the provisions of ‘The Sexual Harassment of Women at Workplace (Prevention, Prohibition and Redressal) Act, 2013, and rules framed thereunder (hereafter referred to as ‘the Act’). Accordingly, while the policy cove

**Q8 — Does JioStar's harassment scope include retaliation for taking part in an investigation?**  
Judge: partly correct, supported = yes  
Answer: Not found in the documents.  
Expected: Yes, the listed scope includes retaliation for reporting or threatening to report harassment, opposing unlawful harassment, or participating in an investigation. It therefore covers adverse responses to raising concerns as well as the underlying unwelcome conduct.  
Answer span in the 3 chunks: **True** → likely cause: **generation**

[1] (jiostar_posh_policy.txt) . J. Confidentiality All information received in the course of the examination and enquiry into a complaint of Sexual Harassment shall be held in trust by the IC and known members of the organization. The identity and addresses of the Aggrieved person, respondent and witnesses, any information relating to conciliation and inquiry proceedings, recommendations of the IC and the action taken by the Company shall not be published, communicated or made known to the public, press and proceedings media in any manner. If any person entrusted with the duty to handle or deal with the complaint, inquiry or any recommendations or actions under law, contravenes his/her confidentiality obligation, he/she shall be liable in accordance with the provisions of the Company policies, as applicable, and may be subjected by the Company to disciplinary action, including but not limited to payment of fines. K. Protection Against Retaliation The Company prohibits employees from ta

**Chosen failure: Q6 (wrong), and the cause is retrieval.** The answer span is clause 4.1.1, *"Responsibilities of Managers: All managers at BFSL must ensure that no employee is subject to harassment …"*. It is in none of the three chunks shown above:
* [1] is the policy's *Purpose & Background*;
* [2] is the sign-off sheet and table of contents, which matched words such as "Policy Guidelines" and "Responsibilities";
* [3] comes from a different organisation's manual (IIM Raipur).

The generator therefore had nothing to compare and correctly replied "Not found in the documents". The needed information was **not** in the chunks, so the error comes from retrieval, not generation.

In Part B, BM25 did not rank the managers' clause in its top 5. Dense search did, at rank 3, but BM25 was the method passed to the reranker, so the cross-encoder never saw that chunk.

**For contrast, Q3 and Q8 are generation failures.** The answer span *was* in the chunks ([2] for Q3, [3] for Q8), yet the 3B generator replied "Not found". In both cases the span was not in chunk [1].

### D2.4 Suggested fix (1 mark)
**Fix: rerank a pool that combines dense and BM25 results, instead of BM25's top 10 alone.** Concretely, pass the RRF-fused top 20 (or simply the union of dense top 10 and BM25 top 10) to the cross-encoder, and keep its top 3.

* **Why it fixes Q6:** dense search ranks the managers'-responsibilities chunk 3rd while BM25 misses it. In a combined pool the cross-encoder would score it against the question, and its wording ("Responsibilities of Managers … ensure … no employee is subject to harassment") matches the question far better than the table-of-contents chunk that BM25 preferred.
* **Why it costs little:** the two methods fail on different queries (Q6 is found only by dense, Q5 is ranked 1st only by BM25), so a combined pool loses nothing that either finds. Reranking 20 candidates instead of 10 adds roughly 20 ms per query, which is negligible next to generation.

For the generation failures (Q3, Q8), a separate fix would add one sentence to the prompt: *"Read all three chunks before deciding the answer is not there."* Alternatively, use the 7B generator.

## Export
`nbconvert` exports the saved `.ipynb`, so save the notebook first (Ctrl+S), or run `python tools/export_html.py` after closing it.

In [ ]:
import subprocess
env = {**os.environ, "JUPYTER_PATH": "/opt/conda/share/jupyter"}
subprocess.run([sys.executable, "-m", "nbconvert", "--to", "html", "notebooks/Group4_Assignment2B_RAG.ipynb",
                "--output-dir", "notebooks"], check=False, env=env)